In [1]:
!pip install -q gradio google-generativeai PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.2 MB/s eta 0:00:00


In [2]:
import gradio as gr
import google.generativeai as genai
import PyPDF2
import os
from getpass import getpass

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
try:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass("🔑 Enter your Google Gemini API Key: ")
    genai.configure(api_key=GOOGLE_API_KEY)
    # Initialize the Gemini 1.5 Flash model
    model = genai.GenerativeModel("gemini-1.5-flash")
    print("✅ Gemini API configured successfully!")
except Exception as e:
    print(f"❌ Error configuring Gemini API: {e}")


🔑 Enter your Google Gemini API Key: ··········
✅ Gemini API configured successfully!


In [6]:
def extract_text_from_pdf(pdf_file):
    """
    Extracts plain text content from an uploaded PDF resume file.
    Args:
        pdf_file: File path of uploaded PDF
    Returns:
        Extracted text string or error message
    """
    try:
        if pdf_file is None:
            return None, "⚠️ Please upload a PDF resume first."

        text = ""
        with open(pdf_file, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            if len(reader.pages) == 0:
                return None, "⚠️ The uploaded PDF appears to be empty."
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

        if not text.strip():
            return None, "⚠️ Could not extract text from the PDF. It might be an image-based or scanned resume."
        return text, None
    except Exception as e:
        return None, f"❌ Error reading PDF: {str(e)}"

In [7]:
def analyze_resume(pdf_file, job_role):
    """
    Analyzes the uploaded resume using Google Gemini 1.5 Flash.
    Returns a structured markdown-formatted analysis.
    """
    # Extract resume text
    resume_text, error = extract_text_from_pdf(pdf_file)
    if error:
        return error

    # Use a default if no target job role is provided
    target_role = job_role.strip() if job_role and job_role.strip() else "General / Best-fit Roles"

    # Carefully crafted prompt for high-quality AI analysis
    prompt = f"""
You are an expert career coach, professional resume reviewer, and ATS (Applicant Tracking System) specialist.

Analyze the following resume in detail and provide a comprehensive evaluation.
The target job role context is: **{target_role}**

Resume Content:
\"\"\"
{resume_text}
\"\"\"

Return your response in clean, well-structured **Markdown** with the following exact sections and emojis:

## 📄 1. Resume Summary
Write a concise 3-4 sentence professional summary of the candidate.

## ✅ 2. Strengths
List 5-7 specific strengths as bullet points. Be precise and reference actual content from the resume.

## ⚠️ 3. Weaknesses
List 4-6 weaknesses or areas that need improvement as bullet points.

## 📊 4. ATS Score
Give a score **out of 100** based on ATS-friendliness (keywords, formatting, structure, clarity, measurable achievements).
Format: **ATS Score: XX/100**
Then add 2-3 sentences explaining the score.

## 🧩 5. Missing Skills
List important skills (technical and soft) that are missing or underrepresented for the target role.

## 💡 6. Improvement Suggestions
Provide 5-7 actionable, specific suggestions to enhance the resume (formatting, wording, quantifiable achievements, keywords, etc.).

## 🎯 7. Best Suitable Job Roles
Suggest 5-7 job titles best suited to this candidate's profile, with a one-line reason for each.

Keep the tone professional, encouraging, and constructive.
"""

    try:
        # Generate AI response from Gemini
        response = model.generate_content(prompt)
        if not response or not response.text:
            return "❌ The AI did not return a response. Please try again."
        return response.text
    except Exception as e:
        return f"❌ Error during AI analysis: {str(e)}"

In [8]:
# ============================================================
# Step 6: Build a Modern Dark-Themed Gradio UI
# ============================================================

# Custom CSS for a polished, professional dark theme
custom_css = """
#main-container {
    max-width: 1100px;
    margin: auto;
    font-family: 'Inter', sans-serif;
}
.gradio-container {
    background: linear-gradient(135deg, #0f0f1e 0%, #1a1a2e 100%) !important;
}
h1 {
    background: linear-gradient(90deg, #00d4ff, #7b61ff);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-weight: 800 !important;
    text-align: center;
}
.title-box {
    text-align: center;
    padding: 20px;
}
.description-box {
    text-align: center;
    color: #c0c0d0;
    font-size: 16px;
    margin-bottom: 20px;
}
button.primary {
    background: linear-gradient(90deg, #00d4ff, #7b61ff) !important;
    border: none !important;
    color: white !important;
    font-weight: 700 !important;
    font-size: 16px !important;
}
footer {visibility: hidden}
"""

# Build the Gradio Blocks interface
with gr.Blocks(theme=gr.themes.Base(primary_hue="indigo", neutral_hue="slate").set(
    body_background_fill="#0f0f1e",
    body_text_color="#e6e6f0",
    block_background_fill="#1a1a2e",
    block_border_color="#2a2a40",
    input_background_fill="#1e1e30",
), css=custom_css, title="AI Resume Analyzer") as demo:

    # Header Section
    gr.HTML("""
        <div class="title-box">
            <h1>🚀 AI Resume Analyzer</h1>
            <p class="description-box">
                Upload your resume PDF and let <b>Google Gemini AI</b> evaluate it like a professional career coach.<br>
                Get an instant <b>ATS Score</b>, strengths, weaknesses, missing skills, and personalized improvement tips.
            </p>
        </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(
                label="📤 Upload Your Resume (PDF)",
                file_types=[".pdf"],
                type="filepath"
            )
            job_role_input = gr.Textbox(
                label="🎯 Target Job Role (Optional)",
                placeholder="e.g., Data Scientist, Frontend Developer, Product Manager",
                lines=1
            )
            analyze_btn = gr.Button("🔍 Analyze My Resume", variant="primary", size="lg")
            gr.Markdown(
                """
                ### 💡 Tips for best results:
                - Upload a **text-based PDF** (not a scanned image).
                - Specify your **target job role** for tailored advice.
                - The AI may take a few seconds to respond.
                """
            )

        with gr.Column(scale=2):
            output_box = gr.Markdown(
                value="### 📝 Your AI-powered resume analysis will appear here...",
                label="Analysis Report"
            )

    # Connect button to analysis function
    analyze_btn.click(
        fn=analyze_resume,
        inputs=[pdf_input, job_role_input],
        outputs=output_box
    )

    # Footer
    gr.HTML("""
        <div style="text-align:center; margin-top:30px; color:#8888aa; font-size:13px;">
            ⚡ Built with Python, Gradio, PyPDF2 & Google Gemini 1.5 Flash
        </div>
    """)

/tmp/ipykernel_9347/495955851.py:43: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base(primary_hue="indigo", neutral_hue="slate").set(
/tmp/ipykernel_9347/495955851.py:43: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base(primary_hue="indigo", neutral_hue="slate").set(


In [9]:
# ============================================================
# Step 7: Launch the Gradio App
# ============================================================
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed963d05ad89243f29.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
